In [2]:
from pydantic import BaseModel, Field, SecretStr, ValidationError
import os
from probill.mcp_servers.mcp_nccn.nccn_tool import DecisionLoader, neo4jDatabase

class Neo4jConfig(BaseModel):
    """Configuration for Neo4j connection"""
    uri: str = Field(default="bolt://localhost:7687", description="Neo4j connection URI")
    user: str = Field(default="neo4j", description="Neo4j username")
    password: str = Field(description="Neo4j password")
    database: str = Field(default="neo4j", description="Neo4j database name")

    @classmethod
    def from_env(cls) -> "Neo4jConfig":
        """Create configuration from environment variables"""
        return cls(
            uri=os.getenv('NEO4J_URI', 'bolt://10.0.40.49:7687'),
            user=os.getenv('NEO4J_USER', 'neo4j'),
            password=os.getenv('NEO4J_PASSWORD', 'sacf_sacf'),
            database=os.getenv('NEO4J_DATABASE', 'neo4j')
        )

async def connect_to_database(config: Neo4jConfig) -> DecisionLoader:
    """Connect to Neo4j database with retry logic"""
    try:
        print(f"Connecting to Neo4j at {config.uri}")
        print(f"Using database {config.database}")
        print(f"Using user {config.user}")
        print(f"Using password {config.password}")

        # db = neo4jDatabase(
        #     neo4j_uri=config.uri, 
        #     neo4j_username=config.user, 
        #     neo4j_password=config.password, 
        #     neo4j_database=config.database
        # )

        # logger.info("Connected to Neo4j database")

        loader = DecisionLoader(
            uri=config.uri,
            user=config.user,
            password=config.password,
            database=config.database
        )
        # Test connection
        loader.driver.verify_connectivity()
        print("Connected to Neo4j database")
        return loader
    except Exception as e:
        print(f"Failed to connect to Neo4j database: {e}")
config = Neo4jConfig.from_env()

loader = await connect_to_database(config)
results = loader.evaluate_patient(patient_id="P010", start_page_id="BINV-20")

print("\n\nResults:","*"*59)
for result in results:
    print(result)

INFO:evaluation:Starting MCP neo4j Server
DEBUG:mcp.server.lowlevel.server:Initializing server 'NCCN MCP Server'
DEBUG:mcp.server.lowlevel.server:Registering handler for ListToolsRequest
DEBUG:mcp.server.lowlevel.server:Registering handler for CallToolRequest
DEBUG:mcp.server.lowlevel.server:Registering handler for ListResourcesRequest
DEBUG:mcp.server.lowlevel.server:Registering handler for ReadResourceRequest
DEBUG:mcp.server.lowlevel.server:Registering handler for PromptListRequest
DEBUG:mcp.server.lowlevel.server:Registering handler for GetPromptRequest
DEBUG:mcp.server.lowlevel.server:Registering handler for ListResourceTemplatesRequest
DEBUG:neo4j.pool:[#0000]  _: <POOL> created, direct address IPv4Address(('10.0.40.49', 7687))
DEBUG:neo4j:[#0000]  _: <WORKSPACE> routing towards fixed database: oncologykg
DEBUG:neo4j:[#0000]  _: <WORKSPACE> pinning database: 'oncologykg'
DEBUG:neo4j.pool:[#0000]  _: <POOL> acquire direct connection, access_mode='READ', database=AcquisitionDatabas

Connecting to Neo4j at bolt://10.0.40.49:7687
Using database oncologykg
Using user neo4j
Using password sacf_sacf
Connected to Neo4j database


DEBUG:neo4j.io:[#EDCC]  S: SUCCESS {'t_first': 238, 'fields': ['data_type', 'value']}
DEBUG:neo4j.io:[#EDCC]  _: <CONNECTION> server state: READY > STREAMING
DEBUG:neo4j.io:[#EDCC]  S: SUCCESS {'bookmark': 'FB:kcwQMCa38HSjRH+X4ha/wAW0NcoAD/7dkA==', 'statuses': [{'gql_status': '02000', 'status_description': 'note: no data'}], 'type': 'r', 't_last': 7, 'db': 'oncologykg'}
DEBUG:neo4j.io:[#EDCC]  _: <CONNECTION> server state: STREAMING > READY
DEBUG:neo4j.pool:[#EDCC]  _: <POOL> released bolt-0
DEBUG:neo4j:[#0000]  _: <WORKSPACE> routing towards fixed database: oncologykg
DEBUG:neo4j.pool:[#0000]  _: <POOL> acquire direct connection, access_mode='WRITE', database=AcquisitionDatabase(name='oncologykg', guessed=False)
DEBUG:neo4j.pool:[#EDCC]  _: <POOL> picked existing connection bolt-0
DEBUG:neo4j.pool:[#EDCC]  _: <POOL> checked re_auth auth=None updated=False force=False
DEBUG:neo4j.pool:[#EDCC]  _: <POOL> handing out existing connection
DEBUG:neo4j.io:[#EDCC]  C: RUN '\n            MATCH



Results: ***********************************************************
{'Page': '{"id":"BINV-20","title":"SYSTEMIC TREATMENT OF RECURRENT UNRESECTABLE (LOCAL OR REGIONAL) OR STAGE IV (M1) DISEASE","page_number":33}'}
{'DecisionPoint': '{"id":"DP-BINV-20-1","question":"Does the patient have bone disease?","answer":"Patient data missed."}'}
{'MissingData': '{\'decision_point\': {\'id\': \'DP-BINV-20-1\', \'question\': \'Does the patient have bone disease?\', \'answer\': \'Patient data missed.\'}, \'missing_data\': [{\'missing_data_id\': \'BoneDiseasePresent\', \'condition_query\': \'UNWIND $patient_data AS data WITH data WHERE data.data_type = "BoneDiseasePresent" RETURN CASE WHEN COUNT(data) = 0 THEN null WHEN HEAD(COLLECT(data.value)) = "true" THEN true ELSE false END AS result\'}]}'}


In [4]:
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client
import logging
# Set up logging
logging.basicConfig(level=logging.DEBUG)
logger = logging.getLogger(__name__)
# Create server parameters for stdio connection
server_params = StdioServerParameters(
    command="mcp-nccn",  # Executable
    args=[],  # Optional command line arguments
    env=None,  # Optional environment variables
)

async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(
            read, write
        ) as session:
            # Initialize the connection
            await session.initialize()
            # List available prompts
            prompts = await session.list_prompts()
            print(type(prompts))
            print(prompts)
            print("Available prompts:")
            for prompt in prompts.prompts:
                print(prompt.name)
                print(prompt.description)
            # Get a prompt
            # prompt = await session.get_prompt(
            #     "evaluation", arguments={"patient_id": "P010", "start_page_id": "BINV-20", "clinical_data": {"HER2": "positive"}}
            # )

            # List available resources
            resources = await session.list_resources()

            # List available tools
            tools = await session.list_tools()
            print("Available tools:")
            for tool in tools.tools:
                print(tool.name)
                print(tool.description)
            # # Read a resource
            # content, mime_type = await session.read_resource("file://some/path")

            # Call a tool
            # result = await session.call_tool(
            #     "evaluate_patient_guidelines", 
            #     arguments={
            #         "patient_id": "P010",
            #         "start_page_id": "BINV-20"
            #     }
            # )
            
            # print(result.content)

await run()

<class 'mcp.types.ListPromptsResult'>
meta=None nextCursor=None prompts=[Prompt(name='evaluation', description='_summary_\n\n    Args:\n        patient_id (str): _description_\n        start_page_id (str): _description_\n        clinical_data (Dict): _description_\n\n    Returns:\n        Dict[str, Any]: _description_\n    ', arguments=[PromptArgument(name='patient_id', description=None, required=True), PromptArgument(name='start_page_id', description=None, required=True), PromptArgument(name='clinical_data', description=None, required=True)])]
Available prompts:
evaluation
_summary_

    Args:
        patient_id (str): _description_
        start_page_id (str): _description_
        clinical_data (Dict): _description_

    Returns:
        Dict[str, Any]: _description_
    
Available tools:
load_nccn_page

    Converts a specified page from a PDF guideline into an image.
    
    Args:
        page (int): Page number to convert from PDF
        
    Returns:
        List: A list conta